<a href="https://colab.research.google.com/github/anamitra-tech/ML-Projects/blob/main/EmotionTrackerFinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gensim


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 52.8 MB/s eta 0:00:00


In [3]:
import os, re, warnings
warnings.filterwarnings("ignore")
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report
from sklearn.impute import SimpleImputer


train_df = pd.read_csv('/content/Sample_arvyax_reflective_dataset.xlsx - Dataset_120.csv')
test_df  = pd.read_csv('/content/arvyax_test_inputs_120.xlsx - Sheet1.csv')

train_df["intensity_3"] = train_df["intensity"].map({1:0, 2:0, 3:1, 4:2, 5:2})

STATES   = ["calm", "focused", "mixed", "neutral", "overwhelmed", "restless"]
AMBIENCE = ["ocean", "forest", "mountain", "rain", "cafe"]
TIMES    = ["morning", "afternoon", "evening", "night", "early_morning"]
FACES    = ["calm_face", "happy_face", "neutral_face", "tired_face", "tense_face", "none", ""]
PMOODS   = ["calm", "focused", "mixed", "neutral", "overwhelmed", "restless", "", "none"]

EVOCAB = {
    "calm":        ["calm","settle","settled","quiet","peaceful","lighter","ease","grounded",
                    "slow","soft","slowed","soften","serene","centered","breathe","breath",
                    "float","release","stillness","pause","relief","less tense"],
    "restless":    ["restless","jumpy","racing","fidgety","scattered","distracted","buzz",
                    "switch","bounce","itchy","unable","wander","still busy","mind jumping",
                    "low buzz","keep wanting"],
    "focused":     ["focus","focused","clear","plan","organize","prioritize","lock",
                    "concentrate","ready","tackle","start","clarity","locked in","sharp",
                    "sharper","next steps"],
    "overwhelmed": ["overwhelmed","overloaded","heavy","pressure","carrying","flooded","piled",
                    "drained","everything","behind","hard","exhausted","too much","drowning",
                    "emotionally tired","want to stop"],
    "neutral":     ["normal","same","steady","average","fine","okay","nothing","fairly",
                    "neutral","aware","not much different","mostly same","just normal","no change"],
    "mixed":       ["mixed","split","between","both","part","two","comforted","distracted",
                    "uneasy","lingering","conflicted","pulled","still uneasy","better but",
                    "not fully","two moods"]
}
VMAP = {m: set(v) for m, v in EVOCAB.items()}


def sem_sim(j):
    if not isinstance(j, str): j = ""
    tok = set(re.findall(r'\b\w+\b', j.lower()))
    return np.array([len(tok & v) / (np.sqrt(len(tok) + 1) * np.sqrt(len(v) + 1))
                     for v in VMAP.values()], dtype=np.float32)


def ambience_proximity(j, a):
    # emotion keywords near the ambience word get proximity boost
    if not isinstance(j, str): j = ""
    if not isinstance(a, str): a = ""
    jl, al = j.lower(), a.lower()
    toks = re.findall(r'\b\w+\b', jl)
    apos = [i for i, t in enumerate(toks) if t == al]
    scores = []
    for kws in EVOCAB.values():
        s = 0.0
        for kw in kws:
            if kw in jl:
                base = 1.0
                if apos:
                    kp = [i for i, t in enumerate(toks) if t == kw.split()[0]]
                    for ap in apos:
                        for k in kp:
                            base = max(base, 2.0 / (1 + abs(ap - k) * 0.1))
                s += base
        scores.append(s)
    tot = sum(scores) + 1e-9
    return np.array([s / tot for s in scores] + [float(al in jl)], dtype=np.float32)


def oh(val, cats):
    v = np.zeros(len(cats), dtype=np.float32)
    s = str(val).lower().strip() if pd.notna(val) else ""
    for i, c in enumerate(cats):
        if c == s: v[i] = 1.0; break
    return v


def sf(row, col, fb=3.0):
    val = row.get(col, fb)
    try: return float(val) if pd.notna(val) else fb
    except: return fb


def build_features(df):
    rows = []
    for _, row in df.iterrows():
        j = row.get("journal_text", "")
        a = row.get("ambience_type", "")
        ss  = sem_sim(j)
        dom = np.zeros(6, dtype=np.float32); dom[np.argmax(ss)] = 1.0
        ss_s = np.sort(ss)[::-1]

        # face × mood outer product (56d) instead of simple concat (15d)
        # this lets the model learn combined signals like tense_face+overwhelmed_prev
        # as a single feature rather than two independent ones
        fv = oh(row.get("face_emotion_hint", ""), FACES)
        pv = oh(row.get("previous_day_mood", ""), PMOODS)
        face_mood_interact = np.outer(fv, pv).flatten()

        # normalize numeric features to [0,1] so they sit on the same scale
        # as the probability-based features (which are already 0-1)
        dur_norm = np.clip(sf(row, "duration_min", 15) / 35.0, 0, 1)
        slp_norm = np.clip(sf(row, "sleep_hours",   6) / 8.5,  0, 1)
        nrg_norm = sf(row, "energy_level", 3) / 5.0
        str_norm = sf(row, "stress_level", 3) / 5.0

        tod_v = oh(row.get("time_of_day", ""), TIMES)
        rq    = {"vague": 0., "conflicted": .5, "clear": 1.}.get(
                    str(row.get("reflection_quality", "vague")).lower().strip(), .25)

        # time_of_day weighted by reflection quality - morning clear session
        # is different signal from morning vague session
        tod_rq = tod_v * rq

        rows.append(np.concatenate([
            ss,                   # 6d  semantic similarity per emotion cluster
            ambience_proximity(j, a),  # 7d  proximity-weighted emotion near ambience
            face_mood_interact,   # 56d face × prev_mood interaction
            oh(a, AMBIENCE),      # 5d  ambience one-hot
            tod_v,                # 5d  time of day one-hot
            np.array([rq], dtype=np.float32),           # 1d reflection quality
            dom,                  # 6d  dominant emotion (argmax of sem_sim)
            np.array([ss_s[0] - ss_s[1]], dtype=np.float32),  # 1d confidence margin
            np.array([dur_norm, slp_norm, nrg_norm, str_norm], dtype=np.float32),  # 4d
            tod_rq,               # 5d  tod × reflection interaction
        ]))
    return np.array(rows, dtype=np.float32)   # 96d structured


def journal_text_only(row):
    # TF-IDF gets journal text ONLY
    # appending ambience/face/mood tokens was found to hurt accuracy by -3.1%
    # because those tokens appear independently of emotion (e.g. "cafe" appears
    # in all cafe sessions regardless of whether the person felt calm or overwhelmed)
    # and TF-IDF boosts their IDF weight, polluting the top SVD components
    j = str(row.get("journal_text", "")) if pd.notna(row.get("journal_text", "")) else ""
    return j


# ------- features -------

print("building features...")

tr_texts = [journal_text_only(r) for _, r in train_df.iterrows()]
te_texts = [journal_text_only(r) for _, r in test_df.iterrows()]

# word bigrams catch phrases like "mind racing", "less tense", "locked in"
# char 4-grams catch typos: "teh" ≈ "the", "tehre" ≈ "there"
# both fit on train only - no leakage
tw = TfidfVectorizer(ngram_range=(1, 2), max_features=500, sublinear_tf=True, min_df=3)
tc = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 4), max_features=200, sublinear_tf=True, min_df=4)

Tw = tw.fit_transform(tr_texts).toarray(); Tw_te = tw.transform(te_texts).toarray()
Tc = tc.fit_transform(tr_texts).toarray(); Tc_te = tc.transform(te_texts).toarray()

# compress sparse tfidf: 500d -> 40d dense, 200d -> 20d dense
sw = TruncatedSVD(min(40, Tw.shape[1] - 1), random_state=42)
sc = TruncatedSVD(min(20, Tc.shape[1] - 1), random_state=42)
Tw = sw.fit_transform(Tw); Tw_te = sw.transform(Tw_te)
Tc = sc.fit_transform(Tc); Tc_te = sc.transform(Tc_te)

Xs    = build_features(train_df)
Xs_te = build_features(test_df)

X_raw    = np.hstack([Tw, Tc, Xs])     # 60d text + 96d structured = 156d
X_raw_te = np.hstack([Tw_te, Tc_te, Xs_te])

imp = SimpleImputer(strategy='median')
sca = StandardScaler()
X    = sca.fit_transform(imp.fit_transform(X_raw))
X_te = sca.transform(imp.transform(X_raw_te))

le = LabelEncoder(); le.classes_ = np.array(STATES)
y1 = le.transform(train_df["emotional_state"].str.lower().str.strip())
y2 = train_df["intensity_3"].values

print(f"feature matrix: {X.shape}")

X_tr, X_val, y1_tr, y1_val, y2_tr, y2_val = train_test_split(
    X, y1, y2, test_size=0.15, random_state=42, stratify=y1
)


# ------- 3-fold CV -------

print("\nrunning 3-fold CV...")
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv1, cv2 = [], []

for fold, (tri, vai) in enumerate(skf.split(X, y1)):
    m1 = HistGradientBoostingClassifier(max_iter=300, max_depth=4, learning_rate=0.05,
                                         min_samples_leaf=10, random_state=42, class_weight="balanced")
    m2 = HistGradientBoostingClassifier(max_iter=300, max_depth=3, learning_rate=0.05,
                                         min_samples_leaf=15, random_state=42)
    m1.fit(X[tri], y1[tri]); m2.fit(X[tri], y2[tri])
    cv1.append(m1.score(X[vai], y1[vai]))
    cv2.append(m2.score(X[vai], y2[vai]))
    print(f"  fold {fold+1}  ->  Y1: {cv1[-1]:.3f}   Y2: {cv2[-1]:.3f}")

print(f"  mean  ->  Y1: {np.mean(cv1):.3f}   Y2: {np.mean(cv2):.3f}")
print(f"  std   ->  Y1: {np.std(cv1):.3f}    Y2: {np.std(cv2):.3f}")


# ------- train final models -------

print("\ntraining...")

clf_emotion = HistGradientBoostingClassifier(
    max_iter=300, max_depth=4, learning_rate=0.05,
    min_samples_leaf=10, random_state=42, class_weight='balanced'
)
clf_emotion.fit(X_tr, y1_tr)
print(f"  emotion val acc : {clf_emotion.score(X_val, y1_val):.3f}")

clf_intensity = HistGradientBoostingClassifier(
    max_iter=300, max_depth=3, learning_rate=0.05,
    min_samples_leaf=15, random_state=42
)
clf_intensity.fit(X_tr, y2_tr)
print(f"  intensity val acc: {clf_intensity.score(X_val, y2_val):.3f}  (3-class, random = 0.33)")


# ------- eval -------

print("\n--- Emotional State ---")
print(classification_report(y1_val, clf_emotion.predict(X_val), target_names=STATES, zero_division=0))

print("--- Intensity (low / medium / high) ---")
print(classification_report(y2_val, clf_intensity.predict(X_val),
      target_names=["low", "medium", "high"], zero_division=0))


# ------- recommendation engine -------
# Q = [cls_probs(6) | intensity_bucket(1) | ambience_oh(5) | dur_norm(1) | slp_norm(1)]
# shape: (14,)  vs previous (7,)
#
# why add ambience and duration:
#   a 4min restless cafe session and a 30min restless ocean session are different
#   situations that should get different recommendations, but the old Q vector
#   gave them identical attention scores
#
# template key matrix K is (8, 14) - each template now has ambience affinity
# and duration affinity baked into its key vector

RECS = [
    {"label": "deep_work",
     "kw": ["focused", "calm", "organized", "clear"],
     "amb": {"ocean": 1.0, "mountain": 0.8, "forest": 0.5, "rain": 0.3, "cafe": 0.5},
     "dur": 0.8,  # suits longer sessions (focused flow needs time)
     "fn": lambda a, t, d, s: f"Your mind is in a receptive state. Use this for deep work. The {a} ambience supported focus today. Start with the hardest task while this clarity holds."},
    {"label": "gentle_reset",
     "kw": ["calm", "settled", "lighter", "peaceful"],
     "amb": {"ocean": 1.0, "forest": 0.8, "mountain": 0.6, "rain": 0.7, "cafe": 0.2},
     "dur": 0.4,
     "fn": lambda a, t, d, s: f"You've settled into a quieter headspace. The {a} ambience anchored this. A short pause or light movement will carry it further into {t}."},
    {"label": "grounding_practice",
     "kw": ["restless", "jumpy", "scattered", "racing"],
     "amb": {"rain": 1.0, "cafe": 0.8, "ocean": 0.5, "forest": 0.5, "mountain": 0.4},
     "dur": 0.3,  # suits short sessions (person couldn't stay still)
     "fn": lambda a, t, d, s: f"Your system is still running fast. The {a} sounds can anchor you - sync your breath slowly. One task at a time will bring the buzz down."},
    {"label": "emotional_offload",
     "kw": ["overwhelmed", "heavy", "flooded", "pressure"],
     "amb": {"forest": 1.0, "rain": 0.9, "ocean": 0.7, "mountain": 0.5, "cafe": 0.2},
     "dur": 0.5,
     "fn": lambda a, t, d, s: f"You're carrying a lot right now. The {a} ambience has been doing quiet work. Try a journal dump or short walk before returning to demands."},
    {"label": "dual_awareness",
     "kw": ["mixed", "split", "between", "uneasy"],
     "amb": {"ocean": 0.8, "mountain": 0.8, "forest": 0.7, "rain": 0.6, "cafe": 0.4},
     "dur": 0.5,
     "fn": lambda a, t, d, s: f"Two emotional currents are running. The {a} setting softened the gap. Don't force resolution - one anchor task will pull you forward."},
    {"label": "steady_continuity",
     "kw": ["neutral", "steady", "same", "fine"],
     "amb": {"cafe": 0.7, "mountain": 0.7, "ocean": 0.6, "rain": 0.5, "forest": 0.6},
     "dur": 0.5,
     "fn": lambda a, t, d, s: f"Your baseline is stable. The {a} session kept things even. Good state for routine work or small creative steps during {t}."},
    {"label": "rest_recovery",
     "kw": ["tired", "tired_face", "drained", "exhausted"],
     "amb": {"ocean": 0.9, "rain": 0.8, "forest": 0.7, "mountain": 0.5, "cafe": 0.2},
     "dur": 0.2,  # suits short sessions (too tired to do a long session)
     "fn": lambda a, t, d, s: f"Fatigue is present. The {a} soundscape offered some softening. Sleep and genuine stillness are the highest-return action right now."},
    {"label": "high_intensity",
     "kw": ["tense_face", "tense", "wound", "unable"],
     "amb": {"rain": 0.8, "ocean": 0.6, "forest": 0.5, "mountain": 0.7, "cafe": 0.5},
     "dur": 0.4,
     "fn": lambda a, t, d, s: f"Physical tension is elevated. Use {a} as a reset between tasks. Break obligations into smaller steps to reduce the felt pressure."},
]

# Q vector dimension: 6 class probs + 1 bucket + 5 ambience + 1 duration + 1 sleep = 14
KDIM = len(STATES) + 1 + len(AMBIENCE) + 1 + 1


def make_key(kws, amb_affinity, dur_affinity):
    k  = np.zeros(KDIM, dtype=np.float32)
    sm = {s: i for i, s in enumerate(STATES)}
    # emotion class affinities from keywords
    for kw in kws:
        for state, idx in sm.items():
            if kw in state or state in kw or kw in VMAP.get(state, set()):
                k[idx] += 1.0
        if kw in ["tired", "tense", "tense_face", "exhausted", "drained"]:
            k[6] += 1.0   # intensity slot
    # ambience affinity (positions 7-11)
    for i, amb in enumerate(AMBIENCE):
        k[7 + i] = amb_affinity.get(amb, 0.0)
    # duration affinity (position 12)
    k[12] = dur_affinity
    # sleep slot (position 13) - left at 0, could be extended
    k[13] = 0.0
    return k / (np.linalg.norm(k) + 1e-9)


KMAT = np.array([make_key(r["kw"], r["amb"], r["dur"]) for r in RECS])


def recommend(probs, intensity_bucket, amb, tod, dur, sleep):
    dur_norm = np.clip(dur / 35.0, 0, 1)
    slp_norm = np.clip(sleep / 8.5, 0, 1)
    amb_oh   = oh(amb, AMBIENCE)

    # extended query: class probs + intensity + ambience + duration + sleep
    Q    = np.concatenate([probs, [intensity_bucket / 2.0], amb_oh, [dur_norm, slp_norm]]).astype(np.float32)
    attn = np.exp(KMAT @ Q / np.sqrt(KDIM))
    attn /= attn.sum()
    rec  = RECS[int(np.argmax(attn))]

    return {
        "recommendation":    rec["fn"](amb, tod, dur, sleep),
        "template":          rec["label"],
        "duration_min":      int(dur),
        "sleep_rec":         8 if sleep < 6 else round(sleep, 1),
        "time_of_day":       tod,
        "attn_weights":      {r["label"]: float(w) for r, w in zip(RECS, attn)},
    }


# ------- run on test set -------

print("\nrunning on test set...")

probs_te  = clf_emotion.predict_proba(X_te)
preds_te  = np.argmax(probs_te, axis=1)
int_te    = clf_intensity.predict(X_te)

state_labels = le.classes_[preds_te]
int_labels   = ["low", "medium", "high"]

rows = []
for i, row in test_df.iterrows():
    idx = i - test_df.index[0]
    amb = str(row.get("ambience_type", "")).lower()
    tod = str(row.get("time_of_day", "")).lower()
    dur = float(row.get("duration_min", 10))
    slp = float(row.get("sleep_hours", 7)) if pd.notna(row.get("sleep_hours")) else 7.0
    rec = recommend(probs_te[idx], int_te[idx], amb, tod, dur, slp)
    rows.append({
        "id":               row["id"],
        "emotional_state":  state_labels[idx],
        "intensity_bucket": int_labels[int_te[idx]],
        "recommendation":   rec["recommendation"],
        "template":         rec["template"],
        "duration_min":     rec["duration_min"],
        "sleep_rec":        rec["sleep_rec"],
        "time_of_day":      rec["time_of_day"],
        "attn_weights":     rec["attn_weights"],
        "confidence":       round(float(probs_te[idx].max()), 3),
        "ambience":         row.get("ambience_type", ""),
    })

out = pd.DataFrame(rows)
print(f"\n{len(out)} predictions done")
print(out["emotional_state"].value_counts().to_string())
print(); print(out["intensity_bucket"].value_counts().to_string())

out.to_csv("arvyax_predictions.csv", index=False)
print("\nsaved -> arvyax_predictions.csv")


building features...
feature matrix: (1200, 156)

running 3-fold CV...
  fold 1  ->  Y1: 0.535   Y2: 0.410
  fold 2  ->  Y1: 0.608   Y2: 0.380
  fold 3  ->  Y1: 0.537   Y2: 0.415
  mean  ->  Y1: 0.560   Y2: 0.402
  std   ->  Y1: 0.034    Y2: 0.015

training...
  emotion val acc : 0.539
  intensity val acc: 0.483  (3-class, random = 0.33)

--- Emotional State ---
              precision    recall  f1-score   support

        calm       0.49      0.56      0.52        32
     focused       0.62      0.55      0.58        29
       mixed       0.55      0.55      0.55        29
     neutral       0.57      0.43      0.49        30
 overwhelmed       0.50      0.55      0.52        29
    restless       0.55      0.58      0.56        31

    accuracy                           0.54       180
   macro avg       0.54      0.54      0.54       180
weighted avg       0.54      0.54      0.54       180

--- Intensity (low / medium / high) ---
              precision    recall  f1-score   suppor